# Korean AutoRAG - William Challenge Evaluation

This notebook demonstrates **Korean AutoRAG** (Marker-Inc-Korea) on the William challenge dataset.

AutoRAG is an open-source framework for automatically optimizing RAG pipelines through AutoML-style automation.

## Prerequisites

1. **Installation**:
```bash
pip install AutoRAG rank-bm25 scikit-learn python-dotenv
```

2. **Data**: William benchmark files in `../lightrag/POC/challenge_data/`
   - `william.md` - 10 interconnected documents
   - `william_benchmark.json` - 22 evaluation questions

**Resources:**
- [Tutorial](https://marker-inc-korea.github.io/AutoRAG/tutorial.html)
- [GitHub](https://github.com/Marker-Inc-Korea/AutoRAG)
- [Documentation](https://marker-inc-korea.github.io/AutoRAG/)

## 1. Installation and Setup

In [1]:
# Install AutoRAG and dependencies
!pip install AutoRAG rank-bm25 scikit-learn python-dotenv --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
import re
from dotenv import load_dotenv

## 2. Prepare Data in AutoRAG Format

AutoRAG requires two parquet files:
- **corpus.parquet**: Document collection (doc_id, contents, metadata)
- **qa.parquet**: Question-answer pairs (qid, query, retrieval_gt, generation_gt)

In [3]:
# Load William benchmark data
data_dir = Path("../lightrag/POC/challenge_data")

# Read the document
with open(data_dir / "william.md", "r") as f:
    full_document = f.read()

# Read benchmark questions
with open(data_dir / "william_benchmark.json", "r") as f:
    benchmark_data = json.load(f)

print(f"Loaded {len(benchmark_data)} benchmark questions")
print(f"Document length: {len(full_document)} characters")

Loaded 22 benchmark questions
Document length: 22197 characters


In [4]:
# Parse the markdown document into separate documents
# William benchmark has 10 documents marked with "## DOCUMENT X:"

# Split by document markers - this creates parts where odd indices are headers
parts = re.split(r"(## DOCUMENT \d+:[^\n]*)", full_document)

documents = []

# Process pairs: header + content
for i in range(1, len(parts), 2):
    if i < len(parts):
        header = parts[i]
        content = parts[i + 1] if i + 1 < len(parts) else ""
        
        # Extract document number
        match = re.search(r"DOCUMENT (\d+):", header)
        if match:
            doc_id = f"doc_{match.group(1)}"
            documents.append({
                "doc_id": doc_id,
                "contents": content.strip()
            })

print(f"\nParsed {len(documents)} documents")
for doc in documents[:3]:
    print(f"  {doc['doc_id']}: {len(doc['contents'])} chars")


Parsed 10 documents
  doc_1: 1202 chars
  doc_2: 5118 chars
  doc_3: 1930 chars


In [5]:
# Create corpus.parquet
corpus_df = pd.DataFrame(documents)
corpus_df['metadata'] = corpus_df['doc_id'].apply(lambda x: {'source': 'william_benchmark'})

print("Corpus DataFrame:")
print(corpus_df.head())
print(f"\nShape: {corpus_df.shape}")

# Save to parquet
os.makedirs("data", exist_ok=True)
corpus_df.to_parquet("data/corpus.parquet", index=False)
print("\n✅ Saved corpus.parquet")


✅ Saved corpus.parquet


In [6]:
# Create qa.parquet
# AutoRAG QA format: qid, query, retrieval_gt (list of relevant doc_ids), generation_gt (answer)

qa_data = []
for i, item in enumerate(benchmark_data):
    qa_data.append({
        "qid": f"q_{i+1}",
        "query": item["question"],
        "retrieval_gt": [[doc_id for doc_id in corpus_df['doc_id'].tolist()]],
        "generation_gt": item["correct_answers"]
    })

qa_df = pd.DataFrame(qa_data)

print("QA DataFrame:")
print(qa_df.head())
print(f"\nShape: {qa_df.shape}")

# Save to parquet
qa_df.to_parquet("data/qa.parquet", index=False)
print("\n✅ Saved qa.parquet")

QA DataFrame:
   qid                                              query  \
0  q_1  Who owns Romano's Bakery, and how is this pers...   
1  q_2  Dr. Okafor treated a patient in the emergency ...   
2  q_3  There are two bakeries mentioned in the docume...   
3  q_4  What significant infrastructure event happened...   
4  q_5  How long has Ahmed Al-Rashid been operating hi...   

                                        retrieval_gt  \
0  [[doc_1, doc_2, doc_3, doc_4, doc_5, doc_6, do...   
1  [[doc_1, doc_2, doc_3, doc_4, doc_5, doc_6, do...   
2  [[doc_1, doc_2, doc_3, doc_4, doc_5, doc_6, do...   
3  [[doc_1, doc_2, doc_3, doc_4, doc_5, doc_6, do...   
4  [[doc_1, doc_2, doc_3, doc_4, doc_5, doc_6, do...   

                                       generation_gt  
0  [Rosa Romano owns/founded Romano's Family Bake...  
1  [Dr. Kwame Okafor (pediatrician) treated Emman...  
2  [Al-Rashid Mediterranean Bakery (run by Ahmed ...  
3  [Bridge 7 was constructed in 1963 (same year R...  
4  [Al-

## 3. Configure OGX Connection

Load OGX endpoint from .env file (shared with LightRAG)

In [27]:
# Load .env file for OGX configuration
env_path = Path("../lightrag/POC/.env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from {env_path}")
else:
    print(f"⚠️  .env not found at {env_path}")
    print("   Copy ../lightrag/POC/.env.example to ../lightrag/POC/.env and configure OGX_BASE_URL and OGX_API_KEY")

CUSTOM_BASE_URL = os.getenv("OGX_BASE_URL")
if not CUSTOM_BASE_URL:
    raise ValueError("OGX_BASE_URL not found in environment variables. Please create .env file.")

OGX_API_KEY = os.getenv("OGX_API_KEY")
if not OGX_API_KEY:
    raise ValueError("OGX_API_KEY not found in environment variables. Please set it in .env file.")

print(f"Using OGX endpoint: {CUSTOM_BASE_URL}")

✅ Loaded .env from ../lightrag/POC/.env
Using OGX endpoint: https://<OGX_BASE_URL>


In [28]:
# Get available models from OGX
from openai import OpenAI

client = OpenAI(
    base_url=f"{CUSTOM_BASE_URL}/v1",
    api_key=OGX_API_KEY
)

model_list = [model.id for model in client.models.list().data]
print(f"Available models: {model_list}")

# Auto-select inference model
inference_models = [m for m in model_list if 'inference' in m or 'llama' in m.lower()]
CUSTOM_MODEL = inference_models[0] if inference_models else model_list[1]
print(f"\nSelected model: {CUSTOM_MODEL}")

# Auto-select embedding model
embedding_models = [m for m in model_list if 'embed' in m.lower()]
EMBEDDING_MODEL = embedding_models[0] if embedding_models else model_list[0]
print(f"Selected embedding model: {EMBEDDING_MODEL}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid JWT token'}}

## 4. Implement Three Retrieval Strategies

We'll manually implement three retrieval methods to compare:
1. **BM25** (lexical/keyword-based)
2. **Vector** (semantic/embedding-based)
3. **Hybrid** (RRF combination of BM25 + Vector)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

def get_embedding(text, model=None):
    if model is None:
        model = EMBEDDING_MODEL
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

# Get document embeddings
print("Generating document embeddings...")
doc_embeddings = []
for doc in documents:
    embedding = get_embedding(doc['contents'])
    doc_embeddings.append(embedding)
    print(f"  {doc['doc_id']}: {len(embedding)} dims")

doc_embeddings = np.array(doc_embeddings)
print(f"\n✅ Generated {len(doc_embeddings)} embeddings")

  doc_9: 1024 dims
  doc_10: 1024 dims

✅ Generated 10 embeddings


In [14]:
# Prepare BM25 index
tokenized_corpus = [doc['contents'].lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ Built BM25 index")

✅ Built BM25 index


In [15]:
def retrieve_bm25(query, top_k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices, scores[top_indices]

def retrieve_vector(query, top_k=5):
    query_embedding = np.array(get_embedding(query)).reshape(1, -1)
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

def retrieve_hybrid(query, top_k=5, alpha=0.5):
    bm25_indices, _ = retrieve_bm25(query, top_k=top_k*2)
    vector_indices, _ = retrieve_vector(query, top_k=top_k*2)

    rrf_scores = {}
    k = 60

    for rank, idx in enumerate(bm25_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + alpha / (k + rank + 1)

    for rank, idx in enumerate(vector_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1 - alpha) / (k + rank + 1)

    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    top_indices = np.array([idx for idx, _ in sorted_items])
    scores = np.array([score for _, score in sorted_items])

    return top_indices, scores

print("✅ Defined retrieval functions")

✅ Defined retrieval functions


In [16]:
def generate_answer(query, retrieved_docs, model=None):
    if model is None:
        model = CUSTOM_MODEL

    context = "\n\n".join([f"Document {i+1}:\n{doc}" for i, doc in enumerate(retrieved_docs)])
    prompt = f"Answer based on context:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

print("✅ Defined generation function")

✅ Defined generation function


## 5. Evaluation Metrics (Same as LightRAG)

- **answer_correctness**: Keyword overlap (Jaccard similarity)
- **faithfulness**: Content overlap between answer and retrieved docs

In [17]:
def calculate_answer_correctness(predicted, ground_truth_list):
    if not predicted or predicted.startswith("Error:"):
        return 0.0

    pred_tokens = set(predicted.lower().split())
    max_overlap = 0.0

    for gt in ground_truth_list:
        gt_tokens = set(gt.lower().split())
        if len(gt_tokens) == 0:
            continue

        intersection = len(pred_tokens & gt_tokens)
        union = len(pred_tokens | gt_tokens)
        overlap = intersection / union if union > 0 else 0
        max_overlap = max(max_overlap, overlap)

    return max_overlap

def calculate_faithfulness(predicted, retrieved_docs):
    if not predicted or predicted.startswith("Error:"):
        return 0.0

    pred_lower = predicted.lower()
    total_overlap = 0

    for doc in retrieved_docs:
        doc_tokens = set(doc.lower().split())
        pred_tokens = set(pred_lower.split())
        overlap = len(doc_tokens & pred_tokens)
        total_overlap += overlap

    pred_length = len(pred_lower.split())
    faithfulness = min(total_overlap / pred_length, 1.0) if pred_length > 0 else 0.0

    return faithfulness

print("✅ Defined evaluation metrics")

✅ Defined evaluation metrics


## 6. Run Evaluation on William Benchmark

This will test all three retrieval methods and calculate initial metrics.

In [18]:
# Run evaluation for all retrieval modes
retrieval_modes = ["bm25", "vector", "hybrid"]
results = []

for mode in retrieval_modes:
    print(f"\n{'='*80}")
    print(f"Evaluating {mode.upper()} retrieval")
    print(f"{'='*80}")

    mode_results = []

    for i, item in enumerate(benchmark_data):
        question = item["question"]
        correct_answers = item["correct_answers"]

        print(f"\nQ{i+1}/{len(benchmark_data)}: {question[:60]}...")

        try:
            if mode == "bm25":
                top_indices, scores = retrieve_bm25(question, top_k=5)
            elif mode == "vector":
                top_indices, scores = retrieve_vector(question, top_k=5)
            else:
                top_indices, scores = retrieve_hybrid(question, top_k=5)

            retrieved_docs = [documents[idx]['contents'] for idx in top_indices]
            predicted_answer = generate_answer(question, retrieved_docs)

            correctness = calculate_answer_correctness(predicted_answer, correct_answers)
            faithfulness = calculate_faithfulness(predicted_answer, retrieved_docs)

            mode_results.append({
                "mode": mode,
                "question": question,
                "predicted": predicted_answer,
                "correct_answers": correct_answers,
                "answer_correctness": correctness,
                "faithfulness": faithfulness
            })

            print(f"  Correctness: {correctness:.3f}, Faithfulness: {faithfulness:.3f}")

        except Exception as e:
            print(f"  Error: {str(e)}")
            mode_results.append({
                "mode": mode,
                "question": question,
                "predicted": f"Error: {str(e)}",
                "correct_answers": correct_answers,
                "answer_correctness": 0.0,
                "faithfulness": 0.0
            })

    results.extend(mode_results)

    avg_correctness = np.mean([r["answer_correctness"] for r in mode_results])
    avg_faithfulness = np.mean([r["faithfulness"] for r in mode_results])
    print(f"\n{mode.upper()} Summary:")
    print(f"  Avg Correctness: {avg_correctness:.3f}")
    print(f"  Avg Faithfulness: {avg_faithfulness:.3f}")

results_df = pd.DataFrame(results)
results_df.to_csv("autorag_evaluation_results.csv", index=False)
print(f"\n💾 Saved detailed results to autorag_evaluation_results.csv")

  Correctness: 0.096, Faithfulness: 0.807

HYBRID Summary:
  Avg Correctness: 0.174
  Avg Faithfulness: 0.904

💾 Saved detailed results to autorag_evaluation_results.csv


## 6.5. LLM-as-a-Judge Evaluation

Use the LLM to judge answer quality on a scale of 1-5 based on:
- Correctness relative to ground truth
- Completeness of the answer
- Relevance to the question

In [21]:
def llm_as_judge(question, predicted_answer, ground_truth_answers, model=None):
    """Use LLM to judge answer quality on scale 1-5"""
    if model is None:
        model = CUSTOM_MODEL
    
    if not predicted_answer or predicted_answer.startswith("Error:"):
        return 1.0  # Minimum score for errors
    
    gt_text = " OR ".join(ground_truth_answers)
    
    judge_prompt = f"""You are an expert evaluator. Rate the quality of the predicted answer compared to the ground truth.

Question: {question}

Ground Truth Answer(s): {gt_text}

Predicted Answer: {predicted_answer}

Rate the predicted answer on a scale of 1-5:
- 5: Perfect answer, matches ground truth completely
- 4: Very good answer, captures main points with minor gaps
- 3: Adequate answer, partially correct but missing key information
- 2: Poor answer, mostly incorrect or incomplete
- 1: Wrong or irrelevant answer

Respond with ONLY a number from 1 to 5."""
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.1,
            max_tokens=10
        )
        
        # Extract numeric score
        score_text = response.choices[0].message.content.strip()
        # Try to extract first number
        import re
        match = re.search(r'[1-5]', score_text)
        if match:
            score = int(match.group())
            return score / 5.0  # Normalize to 0-1 range
        else:
            return 0.5  # Default to middle if can't parse
            
    except Exception as e:
        print(f"  LLM Judge error: {str(e)}")
        return 0.5  # Default to middle on error

print("✅ Defined LLM-as-a-Judge function")

✅ Defined LLM-as-a-Judge function


In [22]:
# Add LLM-as-a-Judge scores to existing results
print("Running LLM-as-a-Judge evaluation...")

# Check if we can load existing results from CSV
if os.path.exists("autorag_evaluation_results.csv"):
    print("📂 Loading existing results from CSV...")
    results_df = pd.read_csv("autorag_evaluation_results.csv")
    results = results_df.to_dict('records')
    print(f"   Loaded {len(results)} existing answers")
    
    # Check if LLM judge scores already exist
    if "llm_judge_score" in results_df.columns:
        print("   ℹ️  LLM judge scores already exist. Re-running evaluation...")
else:
    print("   ⚠️  No existing results found. Please run Section 6 first.")
    print("   This section requires pre-computed answers from the evaluation.")

if results:
    print("\nThis may take a few minutes...")
    
    for i, result in enumerate(results):
        question = result["question"]
        predicted = result["predicted"]
        
        # Handle ground_truth which might be stored as string in CSV
        ground_truth = result["correct_answers"]
        if isinstance(ground_truth, str):
            # Parse if it's a string representation of a list
            import ast
            try:
                ground_truth = ast.literal_eval(ground_truth)
            except:
                ground_truth = [ground_truth]
        
        # Calculate LLM judge score
        llm_score = llm_as_judge(question, predicted, ground_truth)
        result["llm_judge_score"] = llm_score
        
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(results)} answers...")

    # Update DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv("autorag_evaluation_results.csv", index=False)

    print(f"\n✅ Completed LLM-as-a-Judge evaluation")
    print(f"   Average LLM Judge Score: {results_df['llm_judge_score'].mean():.3f}")
else:
    print("\n❌ No results available. Run the evaluation in Section 6 first.")


✅ Completed LLM-as-a-Judge evaluation
   Average LLM Judge Score: 0.821


## 7. Generate Leaderboard

In [23]:
# Calculate leaderboard with LLM-as-a-Judge scores
leaderboard_data = []

for mode in retrieval_modes:
    mode_results = [r for r in results if r["mode"] == mode]

    avg_correctness = np.mean([r["answer_correctness"] for r in mode_results])
    avg_faithfulness = np.mean([r["faithfulness"] for r in mode_results])
    avg_llm_judge = np.mean([r.get("llm_judge_score", 0.0) for r in mode_results])

    leaderboard_data.append({
        "Retrieval Mode": mode.upper(),
        "Answer Correctness": round(avg_correctness, 4),
        "Faithfulness": round(avg_faithfulness, 4),
        "LLM Judge": round(avg_llm_judge, 4),
        "Combined Score": round((avg_correctness + avg_faithfulness + avg_llm_judge) / 3, 4),
        "Num Queries": len(mode_results)
    })

leaderboard_df = pd.DataFrame(leaderboard_data)
leaderboard_df = leaderboard_df.sort_values("Combined Score", ascending=False).reset_index(drop=True)
leaderboard_df.index = leaderboard_df.index + 1
leaderboard_df.index.name = "Rank"

print("\n" + "="*80)
print("KOREAN AUTORAG LEADERBOARD - RETRIEVAL MODE COMPARISON")
print("="*80)
print(leaderboard_df.to_string())
print("\n" + "="*80)

leaderboard_df.to_csv("autorag_leaderboard.csv")
print("\n💾 Saved leaderboard to autorag_leaderboard.csv")


KOREAN AUTORAG LEADERBOARD - RETRIEVAL MODE COMPARISON
     Retrieval Mode  Answer Correctness  Faithfulness  LLM Judge  Combined Score  Num Queries
Rank                                                                                         
1              BM25              0.1786        0.9282     0.8182          0.6417           22
2            HYBRID              0.1736        0.9041     0.8364          0.6380           22
3            VECTOR              0.1650        0.9203     0.8091          0.6315           22


💾 Saved leaderboard to autorag_leaderboard.csv


## 8. Load and Display Saved Leaderboard

In [20]:
# Load saved leaderboard from CSV
if os.path.exists("autorag_leaderboard.csv"):
    saved_leaderboard = pd.read_csv("autorag_leaderboard.csv", index_col="Rank")

    print("="*80)
    print("LOADED LEADERBOARD FROM CSV")
    print("="*80)
    print(saved_leaderboard.to_string())
    print("\n" + "="*80)
else:
    print("❌ No saved leaderboard found. Run the evaluation cells above first.")

LOADED LEADERBOARD FROM CSV
     Retrieval Mode  Answer Correctness  Faithfulness  Combined Score  Num Queries
Rank                                                                              
1              BM25              0.1786        0.9282          0.5534           22
2            VECTOR              0.1650        0.9203          0.5426           22
3            HYBRID              0.1736        0.9041          0.5388           22



## 9. Comparison with LightRAG

Both notebooks use the same William benchmark with the same metrics:
- **answer_correctness**: Keyword overlap (Jaccard similarity)
- **faithfulness**: Content overlap between answer and retrieved docs

**Key Differences:**

| Feature | LightRAG | Korean AutoRAG |
|---------|----------|----------------|
| **Retrieval Modes** | naive, hybrid, mix | bm25, vector, hybrid |
| **Graph Integration** | Yes (local/global graph) | No (traditional retrieval) |
| **Optimization** | Manual configuration | AutoML-style auto-optimization |
| **Storage** | NetworkX, PostgreSQL AGE, Neo4j | Standard vector DB |
| **Best For** | Multi-hop reasoning, graph queries | AutoML pipeline optimization |

**Resources:**
- [AutoRAG Tutorial](https://marker-inc-korea.github.io/AutoRAG/tutorial.html)
- [GitHub Repository](https://github.com/Marker-Inc-Korea/AutoRAG)
- [Documentation](https://marker-inc-korea.github.io/AutoRAG/)